# The Attention Mechanism: Modeling the Nuanced Usage of Words in Language

## Background

In our previous [computational essay](essay.ipynb), we demystified the simplest version of the self-attention mechanism by calculating direct dot products between raw word embeddings. 

We saw how a token like "journey" could act as a query to find its statistical similarity with every other word in the sequence. 

While this baseline approach cleanly introduces the mathematical loop of attention scores, attention weights, and context vectors, it forces a critical limitation: it assumes that a word’s static identity is the same as what it requires to complete meaning, and it offers to other words to complete the meaning of others.

To unlock true linguistic capabilities, we must move beyond these static dictionary representations of words that compress all of a word's properties into a single rigid vector. 

In this computational essay, we look at how modern LLMs use a more sophisticated attention mechanism to model the nuanced usage of words in language.

## Why?

An embedding is a static snapshot of a word containing all of its global dictionary properties. 

Traditional language models treated a word like a fixed entry in a dictionary, keeping its representation rigid. 

However, a word is not always used in the same way. The attention mechanism moves beyond these static definitions, allowing a word's meaning to shift, bend, and adapt based on its neighbors. 

It captures context dynamically by mathematically calculating how much every word in a specific sentence alters the meaning of every other word. 

This gives the model the exact tool it needs to handle multiple related meanings (polysemy) and syntactical understanding that "bank" in a "river bank" and "bank" in a "money bank" are entirely different concepts based on the surrounding tokens.To achieve this dynamic contextualization, a word within any given sentence operates with three distinct structural concerns:

1. What it needs from other words to complete its meaning.
2. What it offers to help other words complete their meanings.
3. What it actually means once its needs and offers are successfully matched with other words.

## Solution
- Concern 1 is modeled by trainable weight matrix W_query.
- Concern 2 is modeled by trainable weight matrix W_key.
- Concern 3 is modeled by trainable weight matrix W_value.

During the training phase, above mentioned matrices as parameters are updated to capture these contextual usage of tokens (words) in the language.

## A Classical Linguistic Parallel

Interestingly, this computational process analogically parallel to classical traditions of language analysis, such as the structural dependency chains found in Classical Arabic grammar (I'rab / إعراب). 

In traditional Arabic analysis, a word never stands alone; it exists within a chain of governance where an active agent (عامل) demands specific structural roles to complete its meaning. This is the concern modeled mathematically by the Query matrix `W_query` in attention mechanism.

At the same time, other words in the sentence broadcast their specific grammatical and structural traits (معمول) to satisfy those requirements, which is modeled by the Key matrix `W_key` in attention mechanism. When these needs and offers match, a structural dependency (تعلق) is established. 

The attention mechanism quantifies this connection to extract and blend the underlying semantic payload—modeled by the Value matrix `W_value` resolving the exact, nuanced role of each word in the sentence.

<img src="images/treebank.png.webp" width="25%"/>

## Goal

Compute the context vector as we did in our simplest implementation version of self attention mechanism.

`context_vectors = attention_mechanism(embedding_vectors, Wk, Wq, Wv)`

## Recipe
- Compute query, key, value vectors of each token by project input embeddings vectors in 3D space unto 2D space.
- Compute attention scores of each token with respect to every other token by using query vector (what it wants from others to complete meaning) and key vectors (what it gives from others to complete meaning).
- Compute attention weights of each token.
- Compute context vectors of each token.

**Note:** _In the following cells, recipe is demystified and understood for one token first and the scaling that understanding to all other tokens._

In [1]:
import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^0)
    [0.55, 0.87, 0.66], # journey  (x^1)
    [0.57, 0.85, 0.64], # starts   (x^2)
    [0.22, 0.58, 0.33], # with     (x^3)
    [0.77, 0.25, 0.10], # one      (x^4)
    [0.05, 0.80, 0.55]  # step     (x^5)
])


d_in = inputs.shape[1] # 3
d_out = 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

print(f"W_query: {W_query}")
print(f"W_key: {W_key}")
print(f"W_value: {W_value}")

W_query: Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]], requires_grad=True)
W_key: Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]], requires_grad=True)
W_value: Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]], requires_grad=True)


## Demystifying: Projecting a Token into a Lower or Equal Dimension Space

To start simple, let's build up the process.

Select one token let's say "journey" or [0.55, 0.87, 0.66] which is currently represent in 3D space.

We need to project this into lower or equal dimensions space let's say 2D space to model the concerns like:
- What **"journey"** as a word needs from other words to complete its meaning i.e. `query_of_second_token`.
- What **"journey"** as a word offers to help other words complete their meanings i.e. `key_of_second_token`.
- What **"journey"** as a word actually means once its needs and offers are successfully matched with other words `value_of_second_token`.

**Note:** _Matrix multiplication is a mathematical way of representing projection of vectors._


In [2]:
x_2 = inputs[1] # "journey"

query_of_second_token = x_2 @ W_query 
key_of_second_token = x_2 @ W_key 
value_of_second_token = x_2 @ W_value

query_of_second_token

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

The goal of attention mechanism is to compute context vectors for all tokens and as we are demystifying the process for one token in this step which needs key and value projecttions for all other tokens to compute the context vector for a specific token (in this case second token "journey").

Therefore, let's compute `keys_of_all_tokens` and `values_of_all_tokens`

In [3]:
keys_of_all_tokens = inputs @ W_key
values_of_all_tokens = inputs @ W_value

print(f"keys_of_all_tokens: {keys_of_all_tokens}")
print(f"values_of_all_tokens: {keys_of_all_tokens}")

keys_of_all_tokens: tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)
values_of_all_tokens: tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)


## Specific Case: Computing Attention Scores -> Attention Weights -> Context Vector with respect to a Token

Following process is the same as previously discussed in simplest implementation version of self attention mechanism [here](essay.ipynb) with a difference that we worked with input embeddings directly but here we will work with these projected vector like `query_of_second_token`.

To start simple, let's build up the process.

Select one token let's say "journey".

Compute the importance of each token in the sequence with respect to this token.
The selected token's projected query vector is used, with which respect to attention score will be calculated with every other token's key vector.

The mechanism to find out the importance or attention score is the dot product of vectors i.e. `dot_product(query_of_second_token, keys_of_all_tokens)` because dot product is measure of similarity between two vectors in our case similarity between tokens:

- dot_product("Your", "journey") = importance of "Your" with respect to "journey"
- dot_product("journey", "journey") = importance of "journey" with respect to "journey"
- dot_product("starts", "journey") = importance of "starts" with respect to "journey"
- dot_product("with", "journey") = importance of "with" with respect to "journey"
- dot_product("one", "journey") = importance of "one" with respect to "journey"
- dot_product("step", "journey") = importance of "step" with respect to "journey"

Here, importance between words means what a given word looks out for in other words and what other words are offering to a given word to complete its meaning.

<img src="images/attention-with-trainable-weights.png" width="50%"/>

In [4]:
d_k = keys_of_all_tokens.shape[1]

attn_scores_wrt_second_token = query_of_second_token @ keys_of_all_tokens.T
attn_weights_wrt_second_token = torch.softmax(attn_scores_wrt_second_token / d_k**0.5, dim=-1)
context_vector_wrt_second_token = attn_weights_wrt_second_token @ values_of_all_tokens

context_vector_wrt_second_token

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

## Generalization: Computing Attention Scores -> Attention Weights -> Context Vector with respect to all Tokens

This was just the attention scores of all tokens with respect to one token and this has to happen for all the tokens with respect to every other token.

In [5]:
queries_of_all_tokens = inputs @ W_query
keys_of_all_tokens = inputs @ W_key
values_of_all_tokens = inputs @ W_value

attn_scores_wrt_all_tokens = queries_of_all_tokens @ keys_of_all_tokens.T
attn_weights_wrt_all_tokens = torch.softmax(attn_scores_wrt_all_tokens / d_k**0.5, dim=-1)
context_vector_wrt_all_tokens = attn_weights_wrt_all_tokens @ values_of_all_tokens

context_vector_wrt_all_tokens

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

**Note:** _Second row is same as previous result we computed with respect to second token only._

## Abstraction of Self Attention (v1)

In [6]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        context_vectors = attn_weights @ values
        
        return context_vectors

## Example: Using the Self Attention Abstraction v1

In [7]:
torch.manual_seed(123)

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^0)
    [0.55, 0.87, 0.66], # journey  (x^1)
    [0.57, 0.85, 0.64], # starts   (x^2)
    [0.22, 0.58, 0.33], # with     (x^3)
    [0.77, 0.25, 0.10], # one      (x^4)
    [0.05, 0.80, 0.55]  # step     (x^5)
])

d_in = inputs.shape[1] # 3
d_out = 2

self_attention_mechanism = SelfAttention_v1(d_in, d_out)

print(self_attention_mechanism(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


**Note:** _Result is the same as we did compute without using the abstraction._

## Improvement: Abstraction of Self Attention (v2)

We can improve the SelfAttention_v1 implementation further by utilizing PyTorch’s nn.Linear layers, which effectively perform matrix multiplication when the bias units are disabled. Additionally, a significant advantage of using nn.Linear instead of manually implementing nn.Parameter(torch.rand(...)) is that nn.Linear has an optimized weight initialization scheme, contributing to more stable and effective model training.

In [8]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        
        context_vectors = attn_weights @ values
        
        return context_vectors

## Example: Using the Self Attention Abstraction v2

In [9]:
torch.manual_seed(789)

inputs = torch.tensor([
    [0.43, 0.15, 0.89], # Your     (x^0)
    [0.55, 0.87, 0.66], # journey  (x^1)
    [0.57, 0.85, 0.64], # starts   (x^2)
    [0.22, 0.58, 0.33], # with     (x^3)
    [0.77, 0.25, 0.10], # one      (x^4)
    [0.05, 0.80, 0.55]  # step     (x^5)
])

d_in = inputs.shape[1] # 3
d_out = 2

self_attention_mechanism = SelfAttention_v2(d_in, d_out)

print(self_attention_mechanism(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Summary
- First we discussed what is the limitation of simple self attention mechanism we discusssed in my last [computational essay](essay.ipynb) and why we need trainable weight matrices for query, key and value.
- We also touched upon how attention mechanism is a mathematical representation of grammatical chain analysis in classical traditions like Arabic. 
- Then we calculated the context vector for a specific token step by step by projecting input vector on to the lower or equal dimension space to capture the nuanced usage of the word within the context.
- Then we generalized the context vector calculations for all tokens in the input sequence.
- Then we abstract the attention mechanism to be easily used in a pluggable fashion.